# Telecom X - Parte 2

## 1.0 Importación de Librerías

In [159]:
# Manejo de warning por importar librerías de la nube
import warnings
warnings.filterwarnings('ignore')

In [160]:
# Librerías que se utilizarán para el desarrollo del modelo
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
#from sklearn.inspection import permutation_importance

## 2.0 Preparación de los Datos

### 2.1 Extracción del Archivo Tratado

In [161]:
# Leer el archivo que se subió a la nube
df = pd.read_csv('/content/datos_tratados.csv')

In [162]:
# Previsualización del DataFrame
df.head()

,Customer_ID,Churn,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges_Monthly,Charges_Total,daily_accounts
0,0002-ORFBO,No,Female,0,1,1,9,1,0,DSL,...,0,1,1,0,One year,1,Mailed check,65.6,593.30,2.186667
1,0003-MKNFE,No,Male,0,0,0,9,1,1,DSL,...,0,0,0,1,Month-to-month,0,Mailed check,59.9,542.40,1.996667
2,0004-TLHLJ,Yes,Male,0,0,0,4,1,0,Fiber optic,...,1,0,0,0,Month-to-month,1,Electronic check,73.9,280.85,2.463333
3,0011-IGKFF,Yes,Male,1,1,0,13,1,0,Fiber optic,...,1,0,1,1,Month-to-month,1,Electronic check,98.0,1237.85,3.266667
4,0013-EXCHZ,Yes,Female,1,1,0,3,1,0,Fiber optic,...,0,1,1,0,Month-to-month,1,Mailed check,83.9,267.40,2.796667


In [163]:
# Lista de las columnas del DataFrame
df.columns

Index(['Customer_ID', 'Churn', 'Gender', 'SeniorCitizen', 'Partner',
       'Dependents', 'Tenure', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'Charges_Monthly', 'Charges_Total',
       'daily_accounts'],
      dtype='object')

In [164]:
# Cramos una Deep Copy
datos = df.copy()

### 2.2 Eliminación de Columnas Irrelevantes

In [165]:
# Eliminar columna de Identificador único
datos = datos.drop(columns= 'Customer_ID')

In [166]:
# Verificar correlación entre posibles columnas redundantes
datos[['Charges_Monthly', 'Charges_Total', 'Tenure', 'daily_accounts']].corr()

,Charges_Monthly,Charges_Total,Tenure,daily_accounts
Charges_Monthly,1.000000,0.652109,0.247982,1.000000
Charges_Total,0.652109,1.000000,0.825118,0.652109
Tenure,0.247982,0.825118,1.000000,0.247982
daily_accounts,1.000000,0.652109,0.247982,1.000000


In [167]:
# Eliminar la columna redundante
datos = datos.drop(columns=['daily_accounts'])

In [168]:
# Verificar cuántos NaN hay en Churn
print(datos['Churn'].isna().sum())

224


In [169]:
# Filtrar los datos que contienen NaN
datos = datos.dropna(subset=['Churn'])

In [170]:
# Verificar si tenemos otros datos NaN
print(datos.isnull().sum())

Churn                0
Gender               0
SeniorCitizen        0
Partner              0
Dependents           0
Tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
Charges_Monthly      0
Charges_Total       11
dtype: int64


In [171]:
# 2. Borrar filas con cualquier valor NaN
datos.dropna(inplace=True)

### 2.3 Encoding

In [172]:
# Convertir 'Churn' a binario
datos['Churn'] = datos['Churn'].map({'Yes': 1, 'No': 0})

# Codificar las variables categóricas
le = LabelEncoder()
for col in datos.select_dtypes(include='object').columns:
    datos[col] = le.fit_transform(datos[col])

# Separar variables y resultado
X = datos.drop('Churn', axis=1)
y = datos['Churn']

# Entrenar Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=666)
model.fit(X, y)

RandomForestClassifier(random_state=666)

In [173]:
# Visualizar nuevo DataFrame
datos

,Churn,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges_Monthly,Charges_Total
0,0,0,0,1,1,9,1,0,0,0,1,0,1,1,0,1,1,3,65.60,593.30
1,0,1,0,0,0,9,1,1,0,0,0,0,0,0,1,0,0,3,59.90,542.40
2,1,1,0,0,0,4,1,0,1,0,0,1,0,0,0,0,1,2,73.90,280.85
3,1,1,1,1,0,13,1,0,1,0,1,1,0,1,1,0,1,2,98.00,1237.85
4,1,0,1,1,0,3,1,0,1,0,0,0,1,1,0,0,1,3,83.90,267.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7262,0,0,0,0,0,13,1,0,0,1,0,0,1,0,0,1,0,3,55.15,742.90
7263,1,1,0,1,0,22,1,1,1,0,0,0,0,0,1,0,1,2,85.10,1873.70
7264,0,1,0,0,0,2,1,0,0,0,1,0,0,0,0,0,1,3,50.30,92.75
7265,0,1,0,1,1,67,1,0,0,1,0,1,1,0,1,2,0,3,67.85,4627.65


### 2.4 Verificación de la Proporción de Cancelación (Churn)

In [174]:
# Mostrar distribución de la variable objetivo 'Churn'
print("Distribución de Churn (conteo):\n", datos['Churn'].value_counts())

Distribución de Churn (conteo):
 Churn
0    5163
1    1869
Name: count, dtype: int64


In [175]:
# Proporción (%)
print("\nProporción de Churn (%):\n", (datos['Churn'].value_counts(normalize=True) * 100).round(2))


Proporción de Churn (%):
 Churn
0    73.42
1    26.58
Name: proportion, dtype: float64


### 2.5 Normalización

In [ ]:
# Aplicamos UnderSampling
from imblearn.under_sampling import RandomUnderSampler

X = datos.drop('Churn', axis=1)
y = datos['Churn']

rus = RandomUnderSampler(random_state=666)
X_res, y_res = rus.fit_resample(X, y)

print("Distribución después de undersampling:\n", y_res.value_counts())

Distribución después de undersampling:
 Churn
0    1869
1    1869
Name: count, dtype: int64


In [ ]:
# Aplicamos OverSampling
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=666)
X_res, y_res = ros.fit_resample(X, y)

print("Distribución después de oversampling:\n", y_res.value_counts())

Distribución después de oversampling:
 Churn
0    5163
1    5163
Name: count, dtype: int64


In [ ]:
# Aplicamos SMOTE
from imblearn.over_sampling import SMOTE
X, y = datos.drop('Churn', axis=1), datos['Churn']

X = X.dropna()
y = y.loc[X.index]

X_res, y_res = SMOTE(random_state=42).fit_resample(X, y)

print("Distribución después de SMOTE:\n", y_res.value_counts())

Distribución después de SMOTE:
 Churn
0    5163
1    5163
Name: count, dtype: int64


In [180]:
from imblearn.over_sampling import SMOTE

X_res, y_res = SMOTE(random_state=66).fit_resample(X, y)

print("Distribución después de SMOTE:\n", y_res.value_counts())

Distribución después de SMOTE:
 Churn
0    5163
1    5163
Name: count, dtype: int64


In [181]:
print(datos.describe())

             Churn       Gender  SeniorCitizen      Partner   Dependents  \
count  7032.000000  7032.000000    7032.000000  7032.000000  7032.000000   
mean      0.265785     0.504693       0.162400     0.482509     0.298493   
std       0.441782     0.500014       0.368844     0.499729     0.457629   
min       0.000000     0.000000       0.000000     0.000000     0.000000   
25%       0.000000     0.000000       0.000000     0.000000     0.000000   
50%       0.000000     1.000000       0.000000     0.000000     0.000000   
75%       1.000000     1.000000       0.000000     1.000000     1.000000   
max       1.000000     1.000000       1.000000     1.000000     1.000000   

            Tenure  PhoneService  MultipleLines  InternetService  \
count  7032.000000   7032.000000    7032.000000      7032.000000   
mean     32.421786      0.903299       0.421928         0.872582   
std      24.545260      0.295571       0.493902         0.737271   
min       1.000000      0.000000       0.00

In [182]:
oversampling = SMOTE()
x_balanceada,y_balanceada = oversampling.fit_resample(X,y)

In [183]:
y_balanceada.value_counts(normalize=True)

,proportion
Churn,
0,0.5
1,0.5


## 3.0 Correlación y Selección de Variables